# 08 — Spark ML: Classification, Regression, and Evaluation

Training a classifier and a regressor with `pyspark.ml`, the prediction/probability column conventions, and the evaluator classes used to score a model — the mechanics an interviewer checks once they know you understand pipelines (notebook 07).

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark MLlib (`pyspark.ml`) — read it as
> a reference and run cell-by-cell once your environment is set up.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

spark = (
    SparkSession.builder.appName("spark-ml-models").master("local[*]")
    .config("spark.sql.shuffle.partitions", "8").getOrCreate()
)

customers = spark.createDataFrame(
    [
        (1, "month-to-month", 29.99, 2, 0), (2, "one-year", 19.50, 24, 0),
        (3, "month-to-month", 89.10, 1, 1), (4, "two-year", 15.00, 36, 0),
        (5, "month-to-month", 95.40, 3, 1), (6, "one-year", 10.00, 18, 0),
        (7, "month-to-month", 75.00, 4, 1), (8, "two-year", 22.00, 30, 0),
        (9, "month-to-month", 60.00, 2, 1), (10, "one-year", 18.00, 20, 0),
    ],
    ["customer_id", "contract_type", "monthly_charges", "tenure_months", "churned"],
)

feature_pipeline = Pipeline(stages=[
    StringIndexer(inputCol="contract_type", outputCol="contract_idx"),
    OneHotEncoder(inputCols=["contract_idx"], outputCols=["contract_vec"]),
    VectorAssembler(inputCols=["contract_vec", "monthly_charges", "tenure_months"],
                    outputCol="features"),
])

train, test = customers.randomSplit([0.8, 0.2], seed=7)
fitted_features = feature_pipeline.fit(train)
train_ready = fitted_features.transform(train)
test_ready = fitted_features.transform(test)

## 1. Column conventions

Every `pyspark.ml` estimator follows the same convention by default: `featuresCol="features"`, `labelCol="label"` (override if your label column is named differently, as here with `churned`). After `.transform()`, a fitted classifier adds:

- **`rawPrediction`** — raw model output (e.g. margin/logit), rarely used directly.
- **`probability`** — calibrated class probabilities (a `Vector`, one entry per class).
- **`prediction`** — the final predicted class label, by default `argmax(probability)` (threshold 0.5 for binary classification).

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="churned", maxIter=20)
lr_model = lr.fit(train_ready)

predictions = lr_model.transform(test_ready)
predictions.select("customer_id", "churned", "rawPrediction", "probability", "prediction").show(truncate=False)

print("coefficients:", lr_model.coefficients)
print("intercept:", lr_model.intercept)

## 2. Tree ensembles: `RandomForestClassifier`

Random forests need no feature scaling, handle non-linear relationships without manual feature crosses, and expose `featureImportances` — a common thing to inspect in an interview follow-up ("which features mattered most?").

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="churned",
                            numTrees=50, maxDepth=5, seed=7)
rf_model = rf.fit(train_ready)
rf_predictions = rf_model.transform(test_ready)
print("feature importances:", rf_model.featureImportances)

## 3. Evaluators — how you actually score a model

- **`BinaryClassificationEvaluator`** — `areaUnderROC` (default metric) or `areaUnderPR`. Operates on `rawPrediction`/`probability`, not the thresholded `prediction` — because ROC/PR curves are computed by sweeping the threshold.
- **`MulticlassClassificationEvaluator`** — `accuracy`, `f1`, `weightedPrecision`, `weightedRecall`. Works for binary too, and is what you use when you need accuracy/F1 specifically. Operates on the thresholded `prediction` column.
- **`RegressionEvaluator`** — `rmse` (default), `mse`, `mae`, `r2`.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(labelCol="churned", metricName="areaUnderROC")
print("AUC:", auc_evaluator.evaluate(predictions))

f1_evaluator = MulticlassClassificationEvaluator(labelCol="churned", predictionCol="prediction", metricName="f1")
print("F1:", f1_evaluator.evaluate(predictions))

acc_evaluator = MulticlassClassificationEvaluator(labelCol="churned", predictionCol="prediction", metricName="accuracy")
print("accuracy:", acc_evaluator.evaluate(predictions))

## 4. Confusion matrix

MLlib doesn't hand you a confusion matrix object directly for DataFrames the way sklearn does — build it with a `groupBy`/`pivot`, the same pattern from notebook 04.

In [ ]:
predictions.groupBy("churned").pivot("prediction").count().orderBy("churned").show()

## 5. Regression: `LinearRegression`

Same shape, different estimator and evaluator — predicting a continuous value (`monthly_charges`) instead of a class.

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

reg_assembler = VectorAssembler(inputCols=["tenure_months"], outputCol="features_reg")
reg_train = reg_assembler.transform(train)
reg_test = reg_assembler.transform(test)

lin_reg = LinearRegression(featuresCol="features_reg", labelCol="monthly_charges")
lin_reg_model = lin_reg.fit(reg_train)
reg_predictions = lin_reg_model.transform(reg_test)

rmse_evaluator = RegressionEvaluator(labelCol="monthly_charges", predictionCol="prediction", metricName="rmse")
r2_evaluator = RegressionEvaluator(labelCol="monthly_charges", predictionCol="prediction", metricName="r2")
print("RMSE:", rmse_evaluator.evaluate(reg_predictions))
print("R2:", r2_evaluator.evaluate(reg_predictions))

## 6. Interview Q&A

1. **"Why does `BinaryClassificationEvaluator` use `rawPrediction` instead of `prediction`?"** — AUC/PR metrics are computed by sweeping the decision threshold across the raw score, not at a single fixed 0.5 cutoff; `prediction` has already collapsed that information.
2. **"How do you fight overfitting in `LogisticRegression`?"** — `regParam` (regularization strength) and `elasticNetParam` (0 = L2 ridge, 1 = L1 lasso, in between = elastic net).
3. **"Why RMSE and not just MAE?"** — RMSE penalizes large errors more heavily (squared term) — appropriate when big misses are disproportionately costly; MAE treats all errors linearly and is more robust to outliers.
4. **"Your accuracy is 95% but the model is useless — why?"** — classic class-imbalance trap (e.g. 95% of customers don't churn, so predicting "never churns" scores 95% accuracy); check F1/AUC/PR curve and the confusion matrix, not raw accuracy, on imbalanced labels.

## Summary

- Fitted classifiers add `rawPrediction`, `probability`, `prediction` columns; regressors add just `prediction`.
- `BinaryClassificationEvaluator` reads raw scores (AUC/PR); `MulticlassClassificationEvaluator` reads the thresholded label (accuracy/F1); `RegressionEvaluator` handles continuous targets.
- On imbalanced classes, don't trust accuracy alone — check F1/AUC and the confusion matrix.
- Next: `09_spark_ml_tuning_and_productionizing.ipynb`.